In [3]:
import math
import random

class Value:
    def __init__(self, data, children=(), op='', label=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self.prev = set(children)
        self.op = op
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        output = Value(self.data + other.data, (self, other), '+')

        def _backward():
            self.grad += 1.0 * output.grad
            other.grad += 1.0 * output.grad
        output._backward = _backward

        return output

    def __radd__(self, other):  # other + self
        return self + other

    def __neg__(self):  # -self
        return self * -1

    def __sub__(self, other):  # self - other
        return self + (-other)

    def __rsub__(self, other):  # other - self
        return other + (-self)

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        output = Value(self.data * other.data, (self, other), '*')

        def _backward():
            self.grad += other.data * output.grad
            other.grad += self.data * output.grad
        output._backward = _backward

        return output

    def __rmul__(self, other):  # other * self
        return self * other

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only supporting int/float powers for now"
        output = Value(self.data ** other, (self,), f'**{other}')

        def _backward():
            self.grad += (other * self.data**(other - 1)) * output.grad
        output._backward = _backward

        return output

    def __truediv__(self, other):  # self / other
        return self * other**-1

    def tanh(self):
        n = self.data
        t = (math.exp(2*n) - 1) / (math.exp(2*n) + 1)
        output = Value(t, (self,), 'tanh')

        def _backward():
            self.grad += (1 - t**2) * output.grad
        output._backward = _backward

        return output

    def backward(self):
        topo = []
        visited = set()

        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v.prev:
                    build_topo(child)
                topo.append(v)

        build_topo(self)
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()


class Neuron:
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1, 1))

    def __call__(self, x):
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        out = act.tanh()
        return out

    def parameters(self):
        return self.w + [self.b]


class Layer:
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs

    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]


class MLP:
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

In [4]:
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0],
]
ys = [1.0, -1.0, -1.0, 1.0]

n = MLP(3, [4, 4, 1])

for step in range(50):
    # forward pass
    ypred = [n(x) for x in xs]
    loss = sum((yp - yt)**2 for yp, yt in zip(ypred, ys))

    # backward pass
    for p in n.parameters():
        p.grad = 0.0  # reset gradients before backward (avoids accumulation across steps)
    loss.backward()

    # update
    for p in n.parameters():
        p.data -= 0.05 * p.grad

    print(step, loss.data)

0 7.808622868271383
1 7.668099418562839
2 7.567511852402948
3 7.40985232591936
4 7.127379454007041
5 6.53920700650224
6 5.14785164917463
7 2.7755939419536544
8 1.5873743292869174
9 0.8364484210503536
10 0.4771827830075586
11 0.31670489079006003
12 0.23176480667591387
13 0.18070840946595987
14 0.1471229550002112
15 0.12354446201052802
16 0.10617021694183564
17 0.09288297126712435
18 0.08241898775174485
19 0.07398098094251002
20 0.0670427923086256
21 0.061244036738821776
22 0.056330053182846423
23 0.052116053761552164
24 0.04846485928448875
25 0.04527259550685605
26 0.042459229075046435
27 0.03996214191697741
28 0.03773166796833564
29 0.03572792942557414
30 0.033918553060033975
31 0.03227699461264715
32 0.030781291027040452
33 0.029413118693231072
34 0.028157073866886193
35 0.027000116621490405
36 0.025931136692940334
37 0.0249406112381714
38 0.024020332648886436
39 0.023163190292939905
40 0.022362994153755842
41 0.02161433130297581
42 0.020912448310462392
43 0.02025315429895472
44 0.019